# Data Preprocess

预处理原始数据

## 导入库

In [ ]:
import os
import ast
import json
import requests
import pandas as pd
from tqdm import tqdm
from urllib.parse import unquote

# 忽略一些警告，使输出简洁（如“Future Warnings”）
import warnings
warnings.filterwarnings('ignore')

## 常量与辅助函数

In [ ]:
# 常量

# 定义目录
RAW_DIR = '../data/raw'
QUESTIONNAIRE_DIR = os.path.join(RAW_DIR, 'questionnaire')
PSYCHOPY_DIR = os.path.join(RAW_DIR, 'psychopy')
PREPROCESSED_DIR = '../data/preprocessed'

# 确保目录存在
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(PSYCHOPY_DIR, exist_ok=True)
os.makedirs(PREPROCESSED_DIR, exist_ok=True)

# PsychoJS文件链接在问卷数据中的列名
PSYCHOPY_COL = 'psychoJS'

# 问卷列名映射
COLUMN_MAP = {
    '作答ID': 'participant_id',
    '用户ID': 'user_id',
    '开始时间': 'start_time',
    '结束时间': 'end_time',
    '作答总时长(秒)': 'total_duration',
    '作答渠道': 'channel',
    '发布ID': 'release_id',
    '问卷发布名称': 'survey_name',
    'IP': 'ip',
    '经度': 'longitude',
    '纬度': 'latitude',
    '省份': 'province',
    '城市': 'city',
    '设备类型': 'device_type',
    '操作系统类型': 'os_type',
    '浏览器类型': 'browser_type',
    '屏幕分辨率': 'screen_res',
    '请选择您的性别。': 'gender',
    '请输入您的年龄。': 'age',
    # PsychoJS数据文件链接
    'PsychoJS实验': PSYCHOPY_COL,
    # 物品熟悉度（共16个）
    '请按您对以下物品的熟悉度进行打分。-罐头': 'fam_can',
    '请按您对以下物品的熟悉度进行打分。-气球': 'fam_balloon',
    '请按您对以下物品的熟悉度进行打分。-勺子': 'fam_spoon',
    '请按您对以下物品的熟悉度进行打分。-雨伞': 'fam_umbrella',
    '请按您对以下物品的熟悉度进行打分。-钥匙': 'fam_key',
    '请按您对以下物品的熟悉度进行打分。-绳子': 'fam_rope',
    '请按您对以下物品的熟悉度进行打分。-砖头': 'fam_brick',
    '请按您对以下物品的熟悉度进行打分。-刀': 'fam_knife',
    '请按您对以下物品的熟悉度进行打分。-衣架': 'fam_hanger',
    '请按您对以下物品的熟悉度进行打分。-盒子': 'fam_box',
    '请按您对以下物品的熟悉度进行打分。-螺丝刀': 'fam_screwdriver',
    '请按您对以下物品的熟悉度进行打分。-袜子': 'fam_sock',
    '请按您对以下物品的熟悉度进行打分。-床单': 'fam_sheet',
    '请按您对以下物品的熟悉度进行打分。-报纸': 'fam_newspaper',
    '请按您对以下物品的熟悉度进行打分。-轮胎': 'fam_tire',
    '请按您对以下物品的熟悉度进行打分。-鞋子': 'fam_shoe',
    # 大五人格（10题）
    '以下是对人的个性的一些描述，请您判断一下每个描述在多大程度上符合您对自己的客观评判。真实回答即可。-1.外向的、热情的': 'bfi_1',
    '以下是对人的个性的一些描述，请您判断一下每个描述在多大程度上符合您对自己的客观评判。真实回答即可。-2.挑剔的、爱争执的': 'bfi_2',
    '以下是对人的个性的一些描述，请您判断一下每个描述在多大程度上符合您对自己的客观评判。真实回答即可。-3.可靠的、自律的': 'bfi_3',
    '以下是对人的个性的一些描述，请您判断一下每个描述在多大程度上符合您对自己的客观评判。真实回答即可。-4.忧虑的、易心烦的?': 'bfi_4',
    '以下是对人的个性的一些描述，请您判断一下每个描述在多大程度上符合您对自己的客观评判。真实回答即可。-5.喜欢新体验的、常有新想法的?': 'bfi_5',
    '以下是对人的个性的一些描述，请您判断一下每个描述在多大程度上符合您对自己的客观评判。真实回答即可。-6.内向的、安静的?': 'bfi_6',
    '以下是对人的个性的一些描述，请您判断一下每个描述在多大程度上符合您对自己的客观评判。真实回答即可。-7.富有同情心的、温暖的?': 'bfi_7',
    '以下是对人的个性的一些描述，请您判断一下每个描述在多大程度上符合您对自己的客观评判。真实回答即可。-8.条理性差的、粗心的?': 'bfi_8',
    '以下是对人的个性的一些描述，请您判断一下每个描述在多大程度上符合您对自己的客观评判。真实回答即可。-9.镇定的、情绪稳定的': 'bfi_9',
    '以下是对人的个性的一些描述，请您判断一下每个描述在多大程度上符合您对自己的客观评判。真实回答即可。-10.遵循常规的、不爱创新的': 'bfi_10',
    # 创意自我效能（4题）
    '请您根据自己的实际感受和体会，对下面4项描述与自身的符合情况进行评价和判断。-1.我对自己运用创意解决问题的能力有信心。': 'cse_1',
    '请您根据自己的实际感受和体会，对下面4项描述与自身的符合情况进行评价和判断。-2.我觉得自己擅长于想出新的点子。': 'cse_2',
    '请您根据自己的实际感受和体会，对下面4项描述与自身的符合情况进行评价和判断。-3.我很擅长从别人的点子中，发展出另一套自己的想法。': 'cse_3',
    '请您根据自己的实际感受和体会，对下面4项描述与自身的符合情况进行评价和判断。-4.对于想出解决问题的新方法，我很拿手。': 'cse_4',
    # 发散思维量表（RIBS，20题）
    '请您判断以下每个描述在多大程度上符合您对自己的客观评判（本题数量较多，后续题目都较少，请耐心作答）。-1.在家里，我会有重新安排摆放家具的想法。': 'ribs_1',
    '请您判断以下每个描述在多大程度上符合您对自己的客观评判（本题数量较多，后续题目都较少，请耐心作答）。-2.我有让工作更轻松的点子。': 'ribs_2',
    '请您判断以下每个描述在多大程度上符合您对自己的客观评判（本题数量较多，后续题目都较少，请耐心作答）。-3.我读其他人写的东西后会意识到还有其它的视角，并对主题有着自己的想法。': 'ribs_3',
    '请您判断以下每个描述在多大程度上符合您对自己的客观评判（本题数量较多，后续题目都较少，请耐心作答）。-4.我对将来要做什么有很多想法。': 'ribs_4',
    '请您判断以下每个描述在多大程度上符合您对自己的客观评判（本题数量较多，后续题目都较少，请耐心作答）。-5.我考虑过做不同的职业。': 'ribs_5',
    '请您判断以下每个描述在多大程度上符合您对自己的客观评判（本题数量较多，后续题目都较少，请耐心作答）。-6.我经常晚上睡不着,因为脑中会有很多点子冒出来，这让我很清醒。': 'ribs_6',
    '请您判断以下每个描述在多大程度上符合您对自己的客观评判（本题数量较多，后续题目都较少，请耐心作答）。-7.当我制定了计划(比如，去特定的餐厅吃饭或去看某场电影)，却被一些事情搞砸后，我能够很容易地找到其它事情来代替。': 'ribs_7',
    '请您判断以下每个描述在多大程度上符合您对自己的客观评判（本题数量较多，后续题目都较少，请耐心作答）。-8.给一场电影或电视节目想一个好的情节，我有很多点子。': 'ribs_8',
    '请您判断以下每个描述在多大程度上符合您对自己的客观评判（本题数量较多，后续题目都较少，请耐心作答）。-9.我有一些关于新发明的点子。': 'ribs_9',
    '请您判断以下每个描述在多大程度上符合您对自己的客观评判（本题数量较多，后续题目都较少，请耐心作答）。-10.我有很多关于创作故事或诗歌的想法。': 'ribs_10',
    '请您判断以下每个描述在多大程度上符合您对自己的客观评判（本题数量较多，后续题目都较少，请耐心作答）。-11.本题请选择“从不”': 'ribs_11',  # 测谎题
    '请您判断以下每个描述在多大程度上符合您对自己的客观评判（本题数量较多，后续题目都较少，请耐心作答）。-12.我能够想出家和学校间的新路线。': 'ribs_12',
    '请您判断以下每个描述在多大程度上符合您对自己的客观评判（本题数量较多，后续题目都较少，请耐心作答）。-13.我有很多关于新的产品或创意的点子。': 'ribs_13',
    '请您判断以下每个描述在多大程度上符合您对自己的客观评判（本题数量较多，后续题目都较少，请耐心作答）。-14.看到云、阴影和类似的模糊图像后我会设想这些形状和图像到底是什么。': 'ribs_14',
    '请您判断以下每个描述在多大程度上符合您对自己的客观评判（本题数量较多，后续题目都较少，请耐心作答）。-15.对于十年后要做什么，我有很多想法': 'ribs_15',
    '请您判断以下每个描述在多大程度上符合您对自己的客观评判（本题数量较多，后续题目都较少，请耐心作答）。-16.在写信时我很难集中在一个主题上因为我会同时想到很多事情。': 'ribs_16',
    '请您判断以下每个描述在多大程度上符合您对自己的客观评判（本题数量较多，后续题目都较少，请耐心作答）。-17.我经常观察人们，并思考他们行为背后的一些其它原因。': 'ribs_17',
    '请您判断以下每个描述在多大程度上符合您对自己的客观评判（本题数量较多，后续题目都较少，请耐心作答）。-18.在看书或看故事时，我能想到更好的结局。': 'ribs_18',
    '请您判断以下每个描述在多大程度上符合您对自己的客观评判（本题数量较多，后续题目都较少，请耐心作答）。-19.当看报纸或某人写的信件时，我经常能想到更好的措辞。': 'ribs_19',
    '请您判断以下每个描述在多大程度上符合您对自己的客观评判（本题数量较多，后续题目都较少，请耐心作答）。-20.听完歌曲我能想到不同或更好的歌词。': 'ribs_20',
    # 任务后评价
    '在完成创造性任务过程中，您所感受到的困难程度如何？': 'difficulty',
    '您觉得刚刚的创造性任务在多大程度上有趣？': 'interest',
    '在创造性任务中，您投入了多少努力来产生想法？': 'effort',
    '对于最终产生的想法，请评估您本人和AI各自贡献的百分比（需在1至100之间选择，且两者之和为100）。-您本人': 'self_contrib',
    '对于最终产生的想法，请评估您本人和AI各自贡献的百分比（需在1至100之间选择，且两者之和为100）。-AI': 'ai_contrib',
    # AI使用原因（多项多选）
    '在创造性任务中，您使用AI辅助的主要原因是什么？-遇到瓶颈，想不出更多想法了': 'reason_bottleneck',
    '在创造性任务中，您使用AI辅助的主要原因是什么？-对当前想法不满意，想换个方向': 'reason_dissatisfied',
    '在创造性任务中，您使用AI辅助的主要原因是什么？-追求独特性，想看AI能否提供更奇特的点子': 'reason_unique',
    '在创造性任务中，您使用AI辅助的主要原因是什么？-验证或对比自己的思路': 'reason_compare',
    '在创造性任务中，您使用AI辅助的主要原因是什么？-纯属好奇，想看看AI会生成什么': 'reason_curiosity',
    '在创造性任务中，您使用AI辅助的主要原因是什么？-其他（请注明）': 'reason_other',
    '在创造性任务中，您使用AI辅助的主要原因是什么？-我没有使用': 'reason_not_use',
    '在创造性任务中，您使用AI辅助的主要原因是什么？-其他（请注明）-文本': 'reason_other_text',
    # AI采用方式
    '对于AI生成的用途建议，您通常在多大程度上会采用以下处理方式？-直接采用AI的建议': 'use_direct',
    '对于AI生成的用途建议，您通常在多大程度上会采用以下处理方式？-对AI的建议进行改写或调整后再采用': 'use_adapt',
    '对于AI生成的用途建议，您通常在多大程度上会采用以下处理方式？-将AI的建议作为跳板，联想到其他新想法': 'use_springboard',
    '对于AI生成的用途建议，您通常在多大程度上会采用以下处理方式？-只是看看，并不采用': 'use_ignore',
    # AI态度
    '请您根据自己的实际感受和体会，对下面3项描述与自身的符合情况进行评价和判断。-1.我对AI技术是接受的。': 'ai_accept',
    '请您根据自己的实际感受和体会，对下面3项描述与自身的符合情况进行评价和判断。-2.人类与AI的关系是和谐的。': 'ai_harmony',
    '请您根据自己的实际感受和体会，对下面3项描述与自身的符合情况进行评价和判断。-3.人类与AI可以很好的合作。': 'ai_cooperate',
    # AI使用频率
    '生成式AI包括但不限于ChatGPT、Deepseek、文心一言、Kimi、豆包、千问等，能够与用户进行自然语言交互、提供信息帮助、执行任务或生成内容的人工智能。请选择最符合你实际情况的一项': 'ai_use_freq',
    '请报告近半年内每个月你使用生成式AI的平均天数': 'ai_days_per_month',
    '创造性活动是指产生新颖、有价值的内容、想法或产品的过程（例如：写作等）。请选择你平时通过生成式AI进行创造性活动的频繁程度。': 'ai_creative_freq'
}

DROPPED_COLUMNS = [
    '用户ID',
    '问卷发布名称',
    '知情同意书研究目的本研究是一项关于创造力的心理学实验，旨在了解在存在AI辅助的情况下，个体在发散思维任务中的创造力表现。测试程序您将首先完成一项创造性思维任务，该过程中需要使用键盘进行输入。任务结束后，您还需填写一份简短问卷。整个测试过程大约需要35分钟。实验开始后可以选择终止，但不可中途暂停，请做好准备在安静房间中使用电脑作答。测试费用本研究不会向您收取任何费用。潜在风险和副作用本研究不涉及任何已知的风险或副作用。受益完成全部测试并通过数据审核后，您将获得20元酬劳。此外，如果您在本批次参与者中的测试表现排名前10%，还将额外获得10元奖励。请注意，数据存在质量差等问题可能遭到拒绝或仅获得部分被试费，还请认真完成实验。隐私本研究的结果可能会在学术期刊/书籍/会议上发表，或者用于教学。但是您的名字或者其他可以确认您身份的信息将不会在任何发表或教学的材料中出现，除非得到您的允许。测试终止您的参与完全基于自愿的原则，您可以在实验的任何过程中退出，并且您不会因为退出实验而受到处罚或损失。',
    '我声明我已经被告知本研究的目的、过程、可能的危险和副作用以及潜在的获益和费用。我的所有问题都得到满意的回答。我已经详细阅读了本知情同意书。我愿意参加本研究。',
    '接下来将进入创造性任务测试。如图所示，您需要等待资源全部下载完毕后，点击“Ok”进入实验程序。如果遇到问题，请尝试关闭浏览器插件或使用无痕浏览模式。[image]',
    '接下来，您需要完成一份问卷调查，预计时间10分钟。所有题目按照自身情况如实填写即可，所有数据仅用于统计分析，将严格保密。被试费将在整个实验完成后一并发放。',
    '本页问题（Q7-Q12）请您根据刚才所进行的4项创造性任务的整体感受来作答。',
]

# 问卷编码映射
FAM_MAP = {'非常不熟悉': 1, '不熟悉': 2, '一般': 3, '熟悉': 4, '非常熟悉': 5}
BFI_MAP = {'完全不同意': 1, '非常不同意': 2, '基本不同意': 3, '不确定': 4, '基本同意': 5, '非常同意': 6, '完全同意': 7}
CSE_MAP = {'非常不同意': 1, '不同意': 2, '不确定': 3, '同意': 4, '非常同意': 5}
RIBS_MAP = {'从不': 1, '很少': 2, '有时': 3, '经常': 4, '总是': 5}
USE_MAP = RIBS_MAP
AI_ATTITUDE_MAP = {'非常不同意': 1, '不同意': 2, '同意': 3, '非常同意': 4}
AI_USE_FREQ_MAP = {
    '我没有听说过生成式AI': 1,
    '我听说过生成式AI ，但我还没有使用过它': 2,
    '我只在一次测试中使用过生成式AI': 3,
    '我用过生成式AI几次，但我不经常使用': 4,
    '我每个月都会使用几次生成式AI': 5,
    '我每周都会使用几次生成式AI': 6,
    '我每天都会使用生成式AI': 7
}

# 定义前缀到映射的字典
PREFIX_MAP = {
    'fam_': FAM_MAP,
    'bfi_': BFI_MAP,
    'cse_': CSE_MAP,
    'ribs_': RIBS_MAP,
    'use_': USE_MAP,
}

# 特殊列单独处理（无法通过前缀区分的）
SPECIAL_MAP = {
    'ai_accept': AI_ATTITUDE_MAP,
    'ai_harmony': AI_ATTITUDE_MAP,
    'ai_cooperate': AI_ATTITUDE_MAP,
    'ai_use_freq': AI_USE_FREQ_MAP,
}

In [ ]:
# 辅助函数

def extract_filename_from_url(url):
    """从URL中提取文件名，并进行URL解码"""
    if pd.isna(url) or not isinstance(url, str):
        return None
    fname = url.split('/')[-1]
    return unquote(fname)

def download_file(url, local_path):
    """下载文件，如果本地已存在则跳过"""
    if os.path.exists(local_path):
        return True
    try:
        response = requests.get(url, timeout=30)
        response.raise_for_status()
        with open(local_path, 'wb') as f:
            f.write(response.content)
        return True
    except Exception as e:
        print(f"下载失败: {url} -> {e}")
        return False

def parse_key_record(keys_str, rt_str, started_time):
    """
    解析按键记录字段，返回按键事件列表。
    参数：
        keys_str: KeyRespRecord.keys 字段的字符串，如 "[""N/A"",""left"",...]"
        rt_str: KeyRespRecord.rt 字段的字符串，如 "[54.96,55.75,...]"
        started_time: 该物品的开始时间（用于转换为相对时间）
    返回：
        list of dict，每个元素包含 'key', 'time'（相对时间）
    """
    if pd.isna(keys_str) or pd.isna(rt_str) or keys_str == '[]' or rt_str == '[]':
        return []
    try:
        keys = ast.literal_eval(keys_str)  # 使用 ast.literal_eval 安全解析列表
        rts = ast.literal_eval(rt_str)
        if len(keys) != len(rts):
            print(f"警告: keys 和 rt 长度不一致，取较短者")
            min_len = min(len(keys), len(rts))
            keys = keys[:min_len]
            rts = rts[:min_len]
        events = []
        for k, t in zip(keys, rts):
            # t 是绝对时间（相对于实验开始），转换为相对该物品开始的时间
            rel_time = float(t) - started_time
            events.append({
                'key': k,
                'time': rel_time
            })
        return events
    except Exception as e:
        print(f"解析按键记录失败: {e}")
        return []

## participants.csv

In [ ]:
# 获取 raw/questionnaire 下的第一个CSV文件
if not os.path.exists(QUESTIONNAIRE_DIR):
    raise FileNotFoundError(f"目录不存在: {QUESTIONNAIRE_DIR}")

csv_files = [f for f in os.listdir(QUESTIONNAIRE_DIR) if f.lower().endswith('.csv')]
if not csv_files:
    raise FileNotFoundError(f"在 {QUESTIONNAIRE_DIR} 中未找到CSV文件")

questionnaire_file = os.path.join(QUESTIONNAIRE_DIR, csv_files[0])
print(f"选取原始问卷文件: {questionnaire_file}")

# 读取问卷数据，跳过第二行（冗余且可能重复的列名）
df_raw = pd.read_csv(questionnaire_file, encoding='gbk', skiprows=[1])
# 丢弃无用列
df_raw.drop(columns=[col for col in DROPPED_COLUMNS if col in df_raw.columns], inplace=True)

# 重命名列
rename_dict = {col: COLUMN_MAP.get(col, col) for col in df_raw.columns if col in COLUMN_MAP}
df_participants = df_raw.rename(columns=rename_dict)

# 遍历所有列进行编码转换
for col in df_participants.columns:
    # 先检查特殊列
    if col in SPECIAL_MAP:
        mapping = SPECIAL_MAP[col]
        df_participants[col] = df_participants[col].map(mapping).fillna(df_participants[col])
        continue

    # 根据前缀匹配
    for prefix, mapping in PREFIX_MAP.items():
        if col.startswith(prefix):
            df_participants[col] = df_participants[col].map(mapping).fillna(df_participants[col])
            break  # 找到匹配后跳出内层循环

# 提取 psychopy 文件名
df_participants['psychopy_file'] = df_participants[PSYCHOPY_COL].apply(extract_filename_from_url)

# 确保 participant_id 存在
df_participants['participant_id'] = range(1, len(df_participants) + 1)

# 保存 participants.csv
participants_out = os.path.join(PREPROCESSED_DIR, 'participants.csv')
df_participants.to_csv(participants_out, index=False, encoding='utf-8-sig')
print(f"已保存 participants.csv，共 {len(df_participants)} 个被试。")

## 下载PsychoPy数据

In [ ]:
# 收集需要下载的文件
file_list = []
for idx, row in df_participants.iterrows():
    pid = row['participant_id']
    fname = row['psychopy_file']
    if pd.isna(fname) or not fname:
        print(f"被试 {pid} 缺少 psychopy 文件名")
        continue
    # 构造本地路径和下载URL
    local_path = os.path.join(PSYCHOPY_DIR, fname)
    orig_row = df_participants.iloc[idx]
    url = orig_row.get(PSYCHOPY_COL, None)
    if pd.isna(url):
        print(f"被试 {pid} 的 psychopy 链接为空")
        continue
    file_list.append((pid, url, local_path))

# 下载文件（带进度条）
print(f"开始下载 {len(file_list)} 个 psychopy 文件...")
success_count = 0
for pid, url, local_path in tqdm(file_list):
    if download_file(url, local_path):
        success_count += 1
    else:
        print(f"被试 {pid} 下载失败: {url}")

print(f"下载完成，成功 {success_count} 个，失败 {len(file_list)-success_count} 个。")

## 处理PsychoPy数据

In [ ]:
# 三个部分的处理函数

def process_messages(trow, pid, item, current_trial, started_time):
    """解析 messages 字段，返回回答事件列表"""
    events = []
    msg_str = trow.get('messages', '')
    if msg_str and msg_str not in ('[]', ''):
        try:
            msgs = json.loads(msg_str)
            for resp_idx, m in enumerate(msgs):
                role = m.get('role', 'unknown')
                content = m.get('content', '')
                time_val = m.get('time', None)
                if time_val is None:
                    continue
                events.append({
                    'participant_id': pid,
                    'item_name': item,
                    'trial_index': current_trial,
                    'response_index': resp_idx + 1,
                    'role': role,
                    'content': content,
                    'time': float(time_val) - started_time
                })
        except Exception as e:
            print(f"解析 messages 失败，被试 {pid}, 物品 {item}: {e}")
    return events

def process_ai_clicks(trow, pid, item, current_trial, started_time):
    """解析 aiClickEvents 字段，返回点击事件列表"""
    events = []
    click_str = trow.get('aiClickEvents', '')
    if click_str and click_str not in ('[]', ''):
        try:
            clicks = json.loads(click_str)
            for click_idx, c in enumerate(clicks):
                time_val = c.get('time', None)
                if time_val is None:
                    continue
                events.append({
                    'participant_id': pid,
                    'item_name': item,
                    'trial_index': current_trial,
                    'click_index': click_idx + 1,
                    'time': float(time_val) - started_time
                })
        except Exception as e:
            print(f"解析 aiClickEvents 失败，被试 {pid}, 物品 {item}: {e}")
    return events

def process_key_record(trow, pid, item, current_trial):
    """解析 KeyRespRecord 字段，返回按键事件列表"""
    events = []
    keys_str = trow.get('KeyRespRecord.keys', '')
    rt_str = trow.get('KeyRespRecord.rt', '')
    if pd.isna(keys_str) or pd.isna(rt_str) or keys_str == '[]' or rt_str == '[]':
        return events
    try:
        keys = ast.literal_eval(keys_str)
        rts = ast.literal_eval(rt_str)
        if len(keys) != len(rts):
            print(f"警告: 被试 {pid} 物品 {item} 的 keys 和 rt 长度不一致，取较短者")
            min_len = min(len(keys), len(rts))
            keys = keys[:min_len]
            rts = rts[:min_len]
        for key_idx, (k, t) in enumerate(zip(keys, rts)):
            events.append({
                'participant_id': pid,
                'item_name': item,
                'trial_index': current_trial,
                'key_index': key_idx + 1,
                'key': k,
                'time': float(t)
            })
    except Exception as e:
        print(f"解析按键记录失败，被试 {pid}, 物品 {item}: {e}")
    return events

In [ ]:
# 初始化空列表
all_responses = []
all_clicks = []
all_keys = []

# 定义练习物品名称（根据数据，练习物品为“肥皂”）
PRACTICE_ITEM = '肥皂'

# 遍历每个有有效文件的被试
for idx, row in df_participants.iterrows():
    pid = row['participant_id']
    fname = row['psychopy_file']
    if pd.isna(fname) or not fname:
        continue
    local_path = os.path.join(PSYCHOPY_DIR, fname)
    if not os.path.exists(local_path):
        print(f"文件不存在: {local_path}")
        continue

    # 读取 psychopy CSV
    try:
        df_psy = pd.read_csv(local_path, encoding='utf-8')
    except Exception as e:
        print(f"读取文件失败 {local_path}: {e}")
        continue

    # 筛选出物品行（itemName 不为空）
    trial_rows = df_psy[df_psy['itemName'].notna()].copy()
    if trial_rows.empty:
        print(f"被试 {pid} 无物品行数据")
        continue

    # 按 AUT.started 排序，确保顺序正确
    trial_rows = trial_rows.sort_values('AUT.started')

    trial_index = 0  # 正式物品序号从1开始，练习为0
    for _, trow in trial_rows.iterrows():
        item = trow['itemName']
        if item == PRACTICE_ITEM:
            current_trial = 0  # 练习
        else:
            trial_index += 1
            current_trial = trial_index

        # 本轮次的开始时间
        started_time = float(trow['AUT.started'])

        # 处理 messages
        all_responses.extend(process_messages(trow, pid, item, current_trial, started_time))
        # 处理 aiClickEvents
        all_clicks.extend(process_ai_clicks(trow, pid, item, current_trial, started_time))
        # 处理 KeyRespRecord
        all_keys.extend(process_key_record(trow, pid, item, current_trial))

print(f"共收集回答 {len(all_responses)} 条，点击 {len(all_clicks)} 条。")

## responses.csv & clicks.csv

In [ ]:
# 保存 responses
if all_responses:
    df_responses = pd.DataFrame(all_responses)
    responses_out = os.path.join(PREPROCESSED_DIR, 'responses.csv')
    df_responses.to_csv(responses_out, index=False, encoding='utf-8-sig')
    print(f"已保存 responses.csv，共 {len(df_responses)} 行。")
else:
    print("无回答数据，responses.csv 未生成。")

# 保存 clicks
if all_clicks:
    df_clicks = pd.DataFrame(all_clicks)
    clicks_out = os.path.join(PREPROCESSED_DIR, 'clicks.csv')
    df_clicks.to_csv(clicks_out, index=False, encoding='utf-8-sig')
    print(f"已保存 clicks.csv，共 {len(df_clicks)} 行。")
else:
    print("无点击数据，clicks.csv 未生成。")

# 保存按键记录
if all_keys:
    df_keys = pd.DataFrame(all_keys)
    keys_out = os.path.join(PREPROCESSED_DIR, 'keys.csv')
    df_keys.to_csv(keys_out, index=False, encoding='utf-8-sig')
    print(f"已保存 keys.csv，共 {len(df_keys)} 行。")
else:
    print("无按键数据，keys.csv 未生成。")